## Proyecto 02- Clasificación Datos BCI

Pamula Huacca Arce | Paula Andrea C. Cano

## 1. Consulta - Selección de índices

- **Índice 1:** Parámetros de Hjorth (Dominio del tiempo)

Los parámetros de Hjorth son un conjunto de indicadores estadísticos que describen las propiedades fundamentales de una señal de electroencefalograma (EEG) directamente en el dominio del tiempo, siendo computacionalmente eficientes al no requerir transformaciones al dominio de la frecuencia. Se componen de tres descriptores principales [1]:

. Actividad (Activity): Mide la varianza de la señal, lo que equivale a la potencia total de la amplitud del segmento de EEG analizado.

. Movilidad (Mobility): Es una estimación de la frecuencia media de la señal. Se calcula como la raíz cuadrada de la relación entre la varianza de la primera derivada de la señal y la varianza de la señal original.

. Complejidad (Complexity): Cuantifica la desviación de la forma de la señal respecto a una onda senoidal pura, indicando la riqueza de los cambios de frecuencia.


En esta segunda parte del proyecto, enfocado en diferenciar estados de reposo e imaginación motora (mano derecha e izquierda), el fenómeno de desincronización relacionada con eventos (ERD) produce cambios morfológicos en las señales captadas por los electrodos motores (C3, Cz y C4). Dado que la Densidad Espectral de Potencia (PSD) extraída en el proyecto anterior se limita al dominio de la frecuencia, la adición de los parámetros de Hjorth (especialmente la Movilidad y la Complejidad) aporta características temporales y variaciones sutiles en la dinámica de la señal. Investigaciones recientes han demostrado empíricamente que la extracción de características de Hjorth, cuando se combina con clasificadores como Máquinas de Soporte Vectorial (SVM), logra discriminar de manera altamente efectiva los movimientos imaginados de las extremidades superiores, superando significativamente a los modelos que solo usan estimaciones de potencia tradicionales [2], [3]. Para este proyecto, estos parámetros se pueden programar fácilmente en Python aplicando operaciones vectorizadas sobre los segmentos de señal usando la librería numpy o mediante funciones predefinidas.


- **Índice 2:** Entropía (Permutación / Shannon) - Medida dinámica no lineal

La entropía de la información (que abarca variantes como la Entropía de Shannon, Entropía Muestral o Entropía de Permutación) es una métrica de la teoría de la información que cuantifica la complejidad, el grado de incertidumbre o el caos de una serie temporal. A diferencia de las métricas tradicionales que asumen linealidad en la señal, la entropía evalúa el comportamiento dinámico no lineal del electroencefalograma (EEG). Un valor de entropía bajo indica que la señal es altamente rítmica, predecible y ordenada, mientras que un valor de entropía alto refleja un patrón más desordenado y complejo con mayor contenido de información [4], [5].


En el contexto de este proyecto de imaginación motora, la entropía resulta ser una característica  útil para entrenar los modelos de Machine Learning. Fisiológicamente, cuando el sujeto se encuentra en estado de reposo, la corteza sensoriomotora (canales C3 y C4) exhibe oscilaciones altamente sincronizadas y rítmicas (ritmos Mu y Beta), lo que se traduce matemáticamente en una entropía baja. Por el contrario, cuando el sujeto imagina el movimiento de una mano, ocurre la desincronización relacionada con eventos (ERD); las neuronas comienzan a disparar de forma independiente, rompiendo el ritmo y haciendo que la señal se vuelva mucho más compleja y caótica. Extraer la entropía permite capturar este aumento repentino de complejidad, compensando las limitaciones de los métodos lineales como la Densidad Espectral de Potencia (PSD) y mejorando significativamente la precisión de clasificación del algoritmo, ya que le otorga una frontera de decisión más clara entre el reposo y la actividad motora [4].



- **Índice 3:** Asimetría de Potencia Interhemisférica (indice de lateralidad espacial )


La asimetría interhemisférica, frecuentemente calculada a través del Índice de Lateralidad (LI), es una métrica espacial que cuantifica la diferencia de actividad eléctrica entre los dos hemisferios del cerebro. En lugar de evaluar cada electrodo de forma aislada, esta medida se obtiene restando o calculando la proporción de la potencia espectral entre un electrodo del hemisferio izquierdo (como C3) y su homólogo en el hemisferio derecho (como C4). Un índice de lateralidad matemático estándar se define mediante la ecuación $LI = (P_{C3} - P_{C4}) / (P_{C3} + P_{C4})$, donde $P$ representa la potencia extraída en una banda específica (como Mu o Beta) para una época determinada [6].

Esta característica espacial es fundamental para este proyecto porque explora directamente el principio fisiológico  de la imaginación motora. Cuando un sujeto imagina el movimiento de la mano derecha, ocurre una desincronización relacionada con eventos (ERD) de forma contralateral; esto significa que la potencia de la banda Mu disminuye significativamente en el electrodo C3 (hemisferio izquierdo), mientras que en el electrodo C4 (hemisferio derecho) la señal se mantiene rítmica o experimenta una sincronización (ERS). Al calcular el índice de asimetría interhemisférica, se consolida este efecto cruzado en una sola variable numérica altamente discriminativa, la cual oscilará hacia valores fuertemente negativos para una clase (imaginación mano derecha) y hacia valores positivos para la otra (imaginación mano izquierda). Esta característica espacial simplifica la frontera de decisión para los algoritmos, lo que incrementará drásticamente el rendimiento clasificatorio de los modelos Máquina de Soporte Vectorial (SVM) y Extreme Gradient Boosting (XGBoost) exigidos en este proyecto [7].



## 2. Plan de análisis y metodología 

Implementaremos un flujo de trabajo estructurado en las siguientes fases metodológicas:

**Fase 1:** Consolidación de datos y características
- Datos EEG y Características: Se tomarán como base las señales limpias y filtradas en la banda de 8 a 30 Hz (ritmos Mu y Beta) provenientes de los canales de interés (C3, Cz y C4) del proyecto 1.  

A cada época de estas señales se les aplicará la extracción de la Densidad Espectral de Potencia (PSD) junto con los tres índices nuevos consultados (Parámetros de Hjorth, Entropía y Asimetría Interhemisférica). Utilizando arreglos vectorizados de numpy  se estructurará la matriz de características final en un DataFrame

**Fase 2:** Exploración y preprocesamiento estadístico
- Exploración : Se generarán diagramas de volin  para evaluar de forma descriptiva cómo varían los índices entre las tres condiciones, seleccionando las representaciones más válidas según la literatura. 
-  Normalización: Se estandarizarán las magnitudes de las características para asegurar que todas las métricas computadas aporten de forma equitativa a los clasificadores.
- Eliminación de correlaciones y selección: Se identificarán y descartarán variables redundantes o con alta correlación para conservar únicamente los índices que muestren mayor diferencia estadística entre las condiciones.
- Análisis Inferencial: Se plantearán las hipótesis nulas ($H_0$) y alternativas ($H_1$) para cada índice, seleccionando la prueba estadística adecuada (paramétrica como ANOVA o no paramétrica como Kruskal-Wallis) para demostrar si las diferencias entre el reposo y las imaginaciones motoras son significativamente válidas. (NO ESTOY SEGURA SI VAMOS A USAR ESA PRUEBAS PARAMETRICAS, HAY QUE DISCUTIRLO :)

**Fase 3:** Estrategia de división y clasificación 
- División (Training/Test 80-20): Los datos con los índices seleccionados se dividirán en un 80% para el entrenamiento  y un 20% para el conjunto de prueba independiente para la evaluación final con datos no vistos.
- Entrenamiento y Evaluación: Se implementarán y compararán tres enfoques de clasificación multiclase 
  1. Redes  Neuronales (MLP): Evaluando por lo menos tres diferentes arquitecturas de red modificando su número de neuronas y capas ocultas.  
  2. Máquina de Soporte Vectorial (SVM).  
  3. Extreme Gradient Boosting (XGBoost).  
- Validación (K-Fold):Para evitar que el modelo se sobreajuste con los datos de entrenamiento y asegurar el funcionamiento adecuado con datos nuevos, usaremos validación cruzada (K-Fold).

**Fase 4:** Evaluación  y métricas de desempeño 
Finalmente, los modelos optimizados se probarán utilizando el conjunto de datos de prueba (20%). Con estos resultados, construiremos y analizaremos las Matrices de Confusión para ver si el algoritmo confunde el reposo con la imaginación de la mano derecha o izquierda. A partir de allí, calcularemos las métricas globales para medir qué tan bien funciona el sistema: Exactitud (Accuracy), Precisión, Sensibilidad (Recall) y F1-Score.



**REFERENCIAS**

[1] A. N. Rodríguez et al., "EEG-BCI Features Discrimination between Executed and Imagined Movements Based on FastICA, Hjorth Parameters, and SVM," Mathematics, vol. 11, no. 21, p. 4409, Nov. 2023, doi: 10.3390/math11214409.
(https://www.mdpi.com/2227-7390/11/21/4409)

[2] M. J. Niegil Francis et al., "EEG-Controlled Robot Navigation using Hjorth Parameters and Welch-PSD," International Journal of Intelligent Engineering and Systems, vol. 14, no. 4, pp. 231-240, 2021, doi: 10.22266/ijies2021.0831.21.
(https://inass.org/wp-content/uploads/2021/07/2021083121.pdf)

[3] X. Yu et al., "Computerized Multidomain EEG Classification System: A New Paradigm," IEEE Journal of Biomedical and Health Informatics, vol. 26, no. 8, pp. 3626-3637, Aug. 2022, doi: 10.1109/JBHI.2022.3168270.
(https://ieeexplore.ieee.org/document/9713960)

[4] https://pubmed.ncbi.nlm.nih.gov/40002501/

[5] https://pubmed.ncbi.nlm.nih.gov/38790441/ 

[6] J. Jin et al., "An Efficient BCI System Based on Motor Imagery and Functional Lateralization," IEEE Transactions on Neural Systems and Rehabilitation Engineering, vol. 30, pp. 248-255, 2022, doi: 10.1109/TNSRE.2022.3146433.
(https://pmc.ncbi.nlm.nih.gov/articles/PMC7697603/)

[7] https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0268880
